In [ ]:
try:
    import openpyxl
    print("openpyxl já está instalado.")
except ImportError:
    print("openpyxl não está instalado, instalando agora...")
    !pip install openpyxl
import numpy as np
import pandas as pd
from google.colab import files

w1 = 2*np.pi*60
w0 = 2*np.pi*250
Q0 = 50
topologia_banco='D'
tensao_fase = 13.8e3/np.sqrt(3)
topologia_banco = 'D'

openpyxl já está instalado.


In [ ]:
def indutor_dessintonia(capacitancia_unidade, ns, w0, topologia_banco):
    C = capacitancia_unidade/ns
    if topologia_banco=='D':
        denominador = 3 * C * w0**2
    elif topologia_banco=='Y':
        denominador = C * w0**2
    indutancia_dessintonia = 1 / denominador
    resistencia_indutor = w0 * indutancia_dessintonia / Q0
    # indutancia_dessintonia=0
    # resistencia_indutor=0
    return [indutancia_dessintonia, resistencia_indutor]

In [ ]:
def grandezas_eletricas_filtro(w, R, L, C, V1, Ih, topologia_banco):
    if topologia_banco == 'Y':
        Zc = 1/(1j*w*C)
    elif topologia_banco == 'D':
        Zc = (1/(1j*w*C))/3
    else:
        print("Algo deu bem errado!")

    Zh = R + 1j*w*L + Zc
    Ih[1] = V1/Zh[1]
    # Tensoes nos elementos série
    VRh = R * Ih
    VLh = 1j*w*L * Ih
    VCh = Zc * Ih
    # Potencia
    SRh = VRh * np.conj(Ih)
    SLh = VLh * np.conj(Ih)
    SCh = VCh * np.conj(Ih)
    # Todos equivalentes em Y
    return [Ih, VRh, VLh, VCh, SRh, SLh, SCh]

In [ ]:
tensao_unidade = 7960
potencia_unidade = 200e3
corrente_unidade = potencia_unidade / tensao_unidade
reatancia_unidade = tensao_unidade / corrente_unidade
capacitancia_unidade = 1 / (w1 * reatancia_unidade)
ns=3 # número de capacitores em série
indutancia_dessintonia, resistencia_indutor = indutor_dessintonia(capacitancia_unidade, ns, w0, topologia_banco)

In [ ]:
f = np.arange(0.0, 61.0, 60.0)
f[0] = 1e-6
w = 2*np.pi*f
Ih = np.zeros(len(w), dtype=complex)

Ih, VRh, VLh, VCh, SRh, SLh, SCh = grandezas_eletricas_filtro(w, resistencia_indutor, indutancia_dessintonia, capacitancia_unidade/ns, tensao_fase, Ih, topologia_banco)

In [ ]:
# Calculando os dados como array 2D onde cada coluna é uma variável
data = np.array([
    np.round(w/w1, 0),  # Harmônico
    np.abs(Ih),  # Corrente
    np.abs(VRh)/tensao_fase,  # Tensão no Resistor
    np.abs(VLh)/tensao_fase,  # Tensão no Indutor
    np.abs(VCh)/tensao_fase,  # Tensão na Unidade Capacitiva
    3*np.real(SRh)/1e3,  # Potência no Resistor
    3*np.imag(SLh)/1e3,  # Potência Reativa no Indutor
    3*np.imag(SCh)/1e3,  # Potência Reativa no Capacitor
    3e-3*(np.abs(np.imag(SLh)+np.imag(SCh))),
    (np.abs(VCh)/ns)/tensao_fase
]).T  # Transposição para alinhar os dados com as colunas

colunas = [('Harmônico', 'h'),
    ('Corrente', 'A'),
    ('Resistor', 'pu'),
    ('Indutor', 'pu'),
    ('Capacitor', 'pu'),
    ('Resistor', 'kW'),
    ('Indutor', 'kVAr'),
    ('Capacitor', 'kVAr'),
    ('Líquidos', 'kVAr'),
    ('Volts Lata', 'pu')
]
multiindex_colunas = pd.MultiIndex.from_tuples(colunas)

df = pd.DataFrame(data, columns=multiindex_colunas)
df_styled = df.style.set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'right')]},  # Corrigindo para "right"
    {'selector': 'td', 'props': [('text-align', 'right')]}  # Alinha dados à direita
]).format("{:.2f}")  # Formata todos os números para duas casas decimais

# Para exibir o DataFrame com o estilo aplicado, simplesmente retorne 'df_styled' em um ambiente Jupyter
df_styled

,Harmônico,Corrente,Resistor,Indutor,Capacitor,Resistor,Indutor,Capacitor,Líquidos,Volts Lata
,h,A,pu,pu,pu,kW,kVAr,kVAr,kVAr,pu
0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,1.00,26.69,0.01,0.06,1.06,3.25,38.99,-676.83,637.85,0.35


In [ ]:
# df_styled.to_excel('meu_arquivo.xlsx', engine='openpyxl')
# files.download('meu_arquivo.xlsx')